# Vantage (full)
Onboard filter + tile codec. 512x512x13 scenes -> 256 tiles (32x32) -> top-K tiles -> 64x6-bit codes.

In [ ]:
%pip install -q torch numpy matplotlib

## 1. Config

In [ ]:
import copy, math, os, time
from dataclasses import dataclass, asdict
import numpy as np
import torch, torch.nn as nn, torch.nn.functional as F
import matplotlib.pyplot as plt

dev = "cuda" if torch.cuda.is_available() else "cpu"
SMOKE = False  # tiny cpu run, pipeline check only
SEED = 0
torch.manual_seed(SEED)

def ftime(s):  # s: seconds -> "1h02m" / "3m05s" / "7s"
    s = int(s); h, r = divmod(s, 3600); m, s = divmod(r, 60)
    return f"{h}h{m:02d}m" if h else (f"{m}m{s:02d}s" if m else f"{s}s")
def log(msg): print(f"▶ {msg}")

@dataclass
class Cfg:
    C: int      # bands
    t: int      # tile side (px)
    g: int      # tiles per image side
    ech: tuple  # tile-enc conv widths (stride 2 each)
    d: int      # embed dim
    nh: int     # attn heads
    nl: int     # transformer layers
    ff: int     # ffn width = d * ff
    cch: int    # codec base width
    lat: int    # codec latent dims per tile
    bits: int   # bits per latent dim

    @property
    def T(self): return self.g ** 2       # tiles per image
    @property
    def H(self): return self.t * self.g   # image side (px)

CFG = Cfg(C=13, t=32, g=16, ech=(32, 64, 128, 256), d=256, nh=8, nl=6, ff=4, cch=32, lat=64, bits=6)
RAW_B = 12   # raw bits per pixel-band
K = 8        # tiles sent per image

# run settings
BS      = 2 if SMOKE else 8        # scenes per step
STEPS   = 40 if SMOKE else 6000
WU      = 5 if SMOKE else 300      # warmup steps
LR      = 5e-4                     # peak lr, cosine to 10%
WD      = 0.01
CLR_M   = 4.0                      # codec lr = LR * CLR_M
PW      = 10.0                     # pos_weight in bce
LAM     = 1.0                      # recon loss weight
CLIP    = 1.0                      # grad-norm clip
LOG_N   = 5 if SMOKE else 100      # log every
EV_EVERY = 20 if SMOKE else 1000   # eval every
EV_N    = 16 if SMOKE else 128     # eval scenes
SV_EVERY = 20 if SMOKE else 500    # ckpt + weights + emb snapshot
SNAP_EVERY = 20 if SMOKE else 2000 # numbered weights copy
EMB_N   = 4 if SMOKE else 8        # fixed probe scenes
EMB_SEED = 4242
OUT     = "vantage_out_smoke" if SMOKE else "vantage_out"
CKPT    = f"{OUT}/checkpoint.pt"
RESUME  = True

log(f"cfg | {dev} | {CFG.C}x{CFG.H}x{CFG.H} | T={CFG.T} | K={K} | SMOKE={SMOKE}")

## 2. Data (synthetic)

In [ ]:
def new_gen(seed): return torch.Generator(device=dev).manual_seed(seed)

def make_batch(n, cfg, gen):
    """-> cur, ref: [n,C,H,H]; y: [n,T] (1 = real event tile, row-major)"""
    dv = gen.device
    C, H, g, t = cfg.C, cfg.H, cfg.g, cfg.t
    ar = torch.arange(H, device=dv, dtype=torch.float32)
    yy, xx = ar[:, None], ar[None, :]                  # [H,1], [1,H]
    sg = torch.Generator().manual_seed(1234)           # fixed spectral signatures
    def sigs(k):                                       # [k,C], norm sqrt(C)
        v = torch.randn(k, C, generator=sg)
        return (v / v.norm(dim=1, keepdim=True) * math.sqrt(C)).to(dv)
    esig, ssig, csig = sigs(3), sigs(1)[0], torch.ones(C, device=dv)  # event x3, static, cloud

    def U(lo, hi): return lo + (hi - lo) * torch.rand(n, generator=gen, device=dv)  # [n]
    def blob(sig, p, sigma, amp):                      # sig: [C] or [n,C]; p: presence prob
        pres = torch.rand(n, generator=gen, device=dv) < p        # [n]
        cy, cx, s, a = U(3, H - 3), U(3, H - 3), U(*sigma), U(*amp)  # center, width, amp: [n]
        d2 = (yy[None] - cy[:, None, None]) ** 2 + (xx[None] - cx[:, None, None]) ** 2  # [n,H,H]
        prof = (a * pres)[:, None, None] * torch.exp(-d2 / (2 * s[:, None, None] ** 2))  # [n,H,H]
        sig = sig if sig.dim() == 2 else sig[None].expand(n, -1)
        return prof[:, None] * sig[:, :, None, None], pres, cy, cx  # [n,C,H,H]
    def field(k, sc):                                  # smooth random field [n,3,H,H] from k x k grid
        z = torch.randn(n, 3, k, k, generator=gen, device=dv)
        return sc * F.interpolate(z, size=(H, H), mode="bilinear", align_corners=False)
    mix = (torch.randn(3, C, generator=sg) / math.sqrt(3)).to(dv)  # [3,C] fixed band mixing

    bg = torch.einsum("nkhw,kc->nchw", field(4, 0.4) + field(32, 0.2), mix)         # [n,C,H,H]
    st = sum(blob(ssig, 0.5, (0.10 * t, 0.20 * t), (2.0, 3.0))[0] for _ in range(6))  # static blobs (both imgs)
    cl = sum(blob(csig, 0.4, (0.08 * H, 0.18 * H), (0.8, 1.2))[0] for _ in range(2))  # clouds (cur only)
    y, ev = torch.zeros(n, cfg.T, device=dv), 0
    for _ in range(4):                                 # event slots
        kind = torch.randint(0, 3, (n,), generator=gen, device=dv)
        out, pres, cy, cx = blob(esig[kind], 0.3, (0.10 * t, 0.17 * t), (2.0, 3.0))
        ev = ev + out
        idx = (cy // t).long() * g + (cx // t).long()  # [n] tile idx of center
        y[torch.arange(n, device=dv)[pres], idx[pres]] = 1.0
    gain = 1 + 0.1 * torch.randn(n, C, 1, 1, generator=gen, device=dv)  # per-band gain on cur
    noise = lambda: 0.05 * torch.randn(n, C, H, H, generator=gen, device=dv)
    ref = bg + st + noise()
    cur = bg * gain + st + ev + cl + noise()
    return cur, ref, y

t0 = time.time()
cur, ref, y = make_batch(4, CFG, new_gen(0))
log(f"data ok ({time.time() - t0:.1f}s, 4 scenes)")
print(f"   scene {tuple(cur.shape[1:])}, {cur[0].numel() * 4 / 1e6:.0f} MB fp32")
print(f"   {y.sum(1).mean():.2f} events/scene = {y.mean() * 100:.2f}% of {CFG.T} tiles")
del cur, ref, y

## 3. Model

In [ ]:
def to_tiles(x, cfg):  # [B,C,H,H] -> [B*T,C,t,t], row-major tile order
    B, C = x.shape[:2]; t = cfg.t
    x = x.unfold(2, t, t).unfold(3, t, t)  # [B,C,g,g,t,t]
    return x.permute(0, 2, 3, 1, 4, 5).reshape(B * cfg.T, C, t, t)

class TileEnc(nn.Module):  # [N,C,t,t] -> [N,d]; shared by cur and ref
    def __init__(s, cfg):
        super().__init__()
        ls, c = [], cfg.C
        for w in cfg.ech:  # each stage halves res: 32->16->8->4->2
            ls += [nn.Conv2d(c, w, 3, 2, 1), nn.BatchNorm2d(w), nn.ReLU()]; c = w
        s.conv = nn.Sequential(*ls)
        s.proj = nn.Linear(c, cfg.d)
    def forward(s, x): return s.proj(s.conv(x).mean((2, 3)))  # gap over 2x2

class PosEmb(nn.Module):  # learned pos emb p: [1,T,d]
    def __init__(s, cfg):
        super().__init__(); s.p = nn.Parameter(torch.randn(1, cfg.T, cfg.d) * 0.02)
    def forward(s, x): return x + s.p

class STE(nn.Module):  # round to 2^bits - 1 levels in fp32, straight-through grad
    def __init__(s, bits): super().__init__(); s.L = 2 ** bits - 1
    def forward(s, z):  # z in [-1,1]
        z = z.float()
        q = torch.round((z + 1) / 2 * s.L) / s.L * 2 - 1
        return z + (q - z).detach()

class Codec(nn.Module):  # enc: [N,C,t,t] -> [N,lat] (quantized); dec: [N,lat] -> [N,C,t,t]
    def __init__(s, cfg):
        super().__init__()
        nd = int(math.log2(cfg.t // 4))  # down to 4x4
        chs = [cfg.cch * 2 ** i for i in range(nd)]
        enc, c = [], cfg.C
        for w in chs: enc += [nn.Conv2d(c, w, 3, 2, 1), nn.ReLU()]; c = w
        s.enc = nn.Sequential(*enc, nn.Flatten(), nn.Linear(c * 16, cfg.lat), nn.Tanh())
        s.q = STE(cfg.bits)
        dec = [nn.Linear(cfg.lat, c * 16), nn.ReLU(), nn.Unflatten(1, (c, 4, 4))]
        rev = chs[::-1] + [cfg.C]
        for i in range(nd):
            dec += [nn.ConvTranspose2d(rev[i], rev[i + 1], 4, 2, 1)] + ([nn.ReLU()] if i < nd - 1 else [])
        s.dec = nn.Sequential(*dec)
    def encode(s, x): return s.q(s.enc(x))
    def forward(s, x): return s.dec(s.encode(x))

class Vantage(nn.Module):
    def __init__(s, cfg):
        super().__init__()
        s.cfg = cfg
        s.enc = TileEnc(cfg)
        s.fuse = nn.Sequential(nn.Linear(3 * cfg.d, cfg.d), nn.GELU(), nn.Linear(cfg.d, cfg.d))  # [ec,er,ec-er] -> d
        s.pos = PosEmb(cfg)
        ly = nn.TransformerEncoderLayer(cfg.d, cfg.nh, cfg.d * cfg.ff, dropout=0.0,
                                        batch_first=True, norm_first=True, activation="gelu")
        s.ctx = nn.TransformerEncoder(ly, cfg.nl, enable_nested_tensor=False)
        s.head = nn.Sequential(nn.LayerNorm(cfg.d), nn.Linear(cfg.d, 1))
        s.codec = Codec(cfg)

    def score(s, cur, ref):  # cur, ref: [B,C,H,H] -> logits [B,T], cur tiles [B*T,C,t,t]
        B, T, d = cur.shape[0], s.cfg.T, s.cfg.d
        tc, tr = to_tiles(cur, s.cfg), to_tiles(ref, s.cfg)
        ec, er = s.enc(tc).view(B, T, d), s.enc(tr).view(B, T, d)
        h = s.ctx(s.pos(s.fuse(torch.cat([ec, er, ec - er], -1))))  # [B,T,d]
        return s.head(h).squeeze(-1), tc

    def embed(s, cur, ref):  # score() + intermediates: ec, er, h [B,T,d]; logits [B,T]; tc
        B, T, d = cur.shape[0], s.cfg.T, s.cfg.d
        tc, tr = to_tiles(cur, s.cfg), to_tiles(ref, s.cfg)
        ec, er = s.enc(tc).view(B, T, d), s.enc(tr).view(B, T, d)
        h = s.ctx(s.pos(s.fuse(torch.cat([ec, er, ec - er], -1))))
        return ec, er, h, s.head(h).squeeze(-1), tc

    @staticmethod
    def select(lg, k, thr=None):  # lg: [B,T] -> bool mask [B,T]; top-k, optionally also p > thr
        m = torch.zeros_like(lg).scatter_(1, lg.topk(k, 1).indices, 1.0).bool()
        return m & (lg.sigmoid() > thr) if thr is not None else m

    def forward(s, cur, ref, k):  # -> logits [B,T], mask [B,T], codes [k*B,lat]
        lg, tiles = s.score(cur, ref)
        m = s.select(lg, k)
        codes = s.codec.encode(tiles[m.view(-1)])  # only selected tiles are encoded
        return lg, m, codes

def parts(cfg):  # param count per child, built on meta device
    with torch.device("meta"):
        m = Vantage(cfg)
    n = lambda mod: sum(p.numel() for p in mod.parameters())
    out = {k: n(v) for k, v in m.named_children() if k != "codec"}
    out["codec.enc"], out["codec.dec"] = n(m.codec.enc), n(m.codec.dec)
    return out

cp = parts(CFG)
ntot, nob = sum(cp.values()), sum(cp.values()) - cp["codec.dec"]  # nob: onboard (dec stays on ground)
for k, v in cp.items(): print(f"   {k:<10}{v:>12,}   ({v / ntot * 100:4.1f}%)")
print(f"   {'total':<10}{ntot:>12,}\n   {'onboard':<10}{nob:>12,}")
print("   onboard MB  " + "  ".join(f"{nm} {nob * b / 8 / 1e6:.2f}" for nm, b in [("fp32", 32), ("fp16", 16), ("int8", 8), ("int4", 4)]))

model = Vantage(CFG).to(dev)
assert sum(p.numel() for p in model.parameters()) == ntot
log(f"model ok: {ntot:,} params on {dev}")

## 4. Train

In [ ]:
amp = dev == "cuda"
adt = torch.bfloat16 if amp and torch.cuda.is_bf16_supported() else torch.float16
ac = lambda: torch.autocast(device_type=dev, dtype=adt, enabled=amp)
gs = torch.amp.GradScaler("cuda", enabled=amp and adt == torch.float16)  # only needed for fp16

isw = lambda n, p: p.dim() >= 2 and not n.startswith("pos")  # weight matrices/kernels get wd
grp = []
for is_c in (False, True):  # codec has its own lr
    for dec in (True, False):
        ps = [p for n, p in model.named_parameters() if n.startswith("codec.") == is_c and isw(n, p) == dec]
        grp.append({"params": ps, "weight_decay": WD if dec else 0.0, "lr": LR * (CLR_M if is_c else 1.0)})
opt = torch.optim.AdamW(grp, lr=LR, betas=(0.9, 0.95))
def lrf(s):  # lr multiplier at step s: linear warmup, then cosine 1 -> 0.1
    if s < WU: return (s + 1) / WU
    pr = (s - WU) / max(1, STEPS - WU)
    return 0.1 + 0.9 * 0.5 * (1 + math.cos(math.pi * min(1.0, pr)))
sch = torch.optim.lr_scheduler.LambdaLR(opt, lrf)
bce = nn.BCEWithLogitsLoss(pos_weight=torch.tensor(PW, device=dev))

def save(obj, path):  # atomic: write tmp then rename
    os.makedirs(os.path.dirname(path), exist_ok=True)
    torch.save(obj, path + ".tmp"); os.replace(path + ".tmp", path)

def save_ckpt(step):  # full resume state
    save({"model": model.state_dict(), "opt": opt.state_dict(), "sched": sch.state_dict(),
          "step": step, "cfg": asdict(CFG)}, CKPT)

def save_w(step, final=False):  # weights only; final also writes onboard/ground split
    sd = {k: v.detach().cpu() for k, v in model.state_dict().items()}
    pk = lambda d: {"step": step, "cfg": asdict(CFG), "state_dict": d}
    save(pk(sd), f"{OUT}/weights_latest.pt")
    if step % SNAP_EVERY == 0 or final: save(pk(sd), f"{OUT}/weights_step{step:06d}.pt")
    if final:
        save(pk({k: v for k, v in sd.items() if not k.startswith("codec.dec.")}), f"{OUT}/onboard_weights.pt")
        save(pk({k: v for k, v in sd.items() if k.startswith("codec.dec.")}), f"{OUT}/ground_decoder.pt")

@torch.no_grad()
def save_emb(step):  # embeddings of fixed probe scenes (same seed every call)
    was = model.training; model.eval()
    gen, P = new_gen(EMB_SEED), {k: [] for k in ("e_cur", "e_ref", "h", "logits", "labels", "codes")}
    for _ in range(math.ceil(EMB_N / BS)):
        cur, ref, y = make_batch(BS, CFG, gen)
        with ac():
            ec, er, h, lg, tc = model.embed(cur, ref)  # ec, er, h: [BS,T,d]; lg: [BS,T]
            q = model.codec.q(model.codec.enc(tc))     # [BS*T,lat] in [-1,1]
        codes = torch.round((q + 1) / 2 * model.codec.q.L).to(torch.uint8 if CFG.bits <= 8 else torch.int16)
        for k, v in zip(P, (ec, er, h, lg, y, codes.view(BS, CFG.T, -1))):
            P[k].append(v.detach().cpu() if k == "codes" else v.detach().float().cpu())
    out = {k: torch.cat(v)[:EMB_N] for k, v in P.items()}  # first dim: EMB_N
    out["labels"] = out["labels"].to(torch.uint8)
    out.update(pos_embedding=model.pos.p.detach().float().cpu(), step=step, probe_seed=EMB_SEED, cfg=asdict(CFG))
    save(out, f"{OUT}/embeddings/emb_step{step:06d}.pt")
    model.train(was)

def save_all(step, final=False):
    save_ckpt(step); save_w(step, final); save_emb(step)

def ls_out():
    print(f"saved files in {OUT}/")
    for root, _, fs in sorted(os.walk(OUT)):
        for f in sorted(fs):
            p = os.path.join(root, f); print(f"   {p:<52}{os.path.getsize(p) / 1e6:8.1f} MB")

@torch.no_grad()
def evl(model, cfg=CFG, n=EV_N, bs=BS, k=K, seed=12345, thr=0.5):
    """streaming eval. rk: recall@k; rkt: recall@k & p>thr; nsent: tiles/scene sent; r2: codec R2; red: raw bits / sent bits"""
    model.eval()
    gen, nb = new_gen(seed), math.ceil(n / bs)
    npos = tpk = tpt = nsent = sse = sx = sx2 = cnt = 0.0
    for _ in range(nb):
        cur, ref, y = make_batch(bs, cfg, gen)
        with ac(): lg, tiles = model.score(cur, ref)
        lg = lg.float()
        mk, mt = Vantage.select(lg, k), Vantage.select(lg, k, thr)  # [bs,T] bool
        npos += y.sum().item(); tpk += (mk.float() * y).sum().item()
        tpt += (mt.float() * y).sum().item(); nsent += mt.sum().item()
        x = tiles[mk.view(-1)]  # [bs*k,C,t,t]
        with ac(): rec = model.codec(x)
        x, rec = x.float(), rec.float()
        sse += ((rec - x) ** 2).sum().item(); sx += x.sum().item(); sx2 += (x ** 2).sum().item(); cnt += x.numel()
    var = sx2 / cnt - (sx / cnt) ** 2
    ns = nsent / (nb * bs)
    tb = cfg.lat * cfg.bits + math.ceil(math.log2(cfg.T))  # bits per sent tile (code + position)
    return {"rk": tpk / max(npos, 1), "rkt": tpt / max(npos, 1), "nsent": ns,
            "r2": 1 - (sse / cnt) / var,
            "red": cfg.C * cfg.H ** 2 * RAW_B / (ns * tb) if ns > 0 else float("inf")}

def say(r, k=K, cfg=CFG, ind="   "):  # plain-words eval summary
    print(f"{ind}finds {r['rk'] * 100:.0f}% of real events in top {k} (random: {k / cfg.T * 100:.0f}%)")
    print(f"{ind}with cutoff: sends {r['nsent']:.1f}/{cfg.T} tiles, finds {r['rkt'] * 100:.0f}%")
    r2 = r["r2"]
    print(f"{ind}decoded tiles recover {r2 * 100:.0f}% of detail" if r2 >= 0 else f"{ind}codec worse than flat average (undertrained)")
    red = r["red"]
    print(f"{ind}downlink: ~{red:,.0f}x less than raw" if math.isfinite(red) else f"{ind}downlink: nothing sent")

def show(r, k=K, cfg=CFG):
    print(f"recall@{k}            {r['rk']:.3f}   (random {k / cfg.T:.3f})")
    print(f"recall@{k}+thr        {r['rkt']:.3f}   avg tiles sent {r['nsent']:.2f}/{cfg.T}")
    print(f"recon R2            {r['r2']:.3f}")
    print(f"downlink reduction  {r['red']:.0f}x   (toy data)")
    say(r)

def plan(start):
    a = f"{str(adt).split('.')[-1]} autocast" if amp else "fp32"
    print("RUN PLAN")
    print(f"  model     {ntot:,} params ({nob:,} onboard) on {dev}, {a}")
    print(f"  data      {BS} scenes/step = {BS * CFG.T * 2:,} tiles through enc; fresh scenes every step")
    print(f"  schedule  {STEPS - start:,} steps ({(STEPS - start) * BS:,} scenes), warmup {WU}, lr {LR:g} (codec x{CLR_M:g}), cosine to 10%")
    print(f"  budget    K = {K}/{CFG.T}")
    print(f"  log       every {LOG_N} steps, eval every {EV_EVERY}")
    print(f"  save      every {SV_EVERY} steps -> {OUT}/ (numbered weights every {SNAP_EVERY})\n")

def train():
    start = 0
    if RESUME and os.path.exists(CKPT):
        ck = torch.load(CKPT, map_location=dev)
        model.load_state_dict(ck["model"]); opt.load_state_dict(ck["opt"]); sch.load_state_dict(ck["sched"])
        start = ck["step"]; log(f"resumed from {CKPT} at step {start}")
    if start >= STEPS: log("already at STEPS"); return
    plan(start)
    arr = lambda new, old: "" if old is None else ("↓" if new < old * 0.98 else "↑" if new > old * 1.02 else "→")
    gen = new_gen(SEED + 1 + start)  # seeded by start step: resume doesn't replay scenes
    model.train(); t0 = time.time()
    acc, na, prev, gem = torch.zeros(3, device=dev), 0, None, None  # acc: [l_sig, l_rec, gnorm] sums; gem: gnorm ema
    for s in range(start, STEPS):
        cur, ref, y = make_batch(BS, CFG, gen)
        with ac():
            lg, tiles = model.score(cur, ref)  # lg: [BS,T]; tiles: [BS*T,C,t,t]
            rec = model.codec(tiles)           # same shape as tiles
        ls, lr_ = bce(lg.float(), y), F.mse_loss(rec.float(), tiles.float())
        loss = ls + LAM * lr_
        if not torch.isfinite(loss): raise RuntimeError(f"non-finite loss at step {s}")
        opt.zero_grad(set_to_none=True)
        gs.scale(loss).backward()
        gs.unscale_(opt)
        gn = nn.utils.clip_grad_norm_(model.parameters(), CLIP)  # pre-clip norm
        gs.step(opt); gs.update(); sch.step()
        acc += torch.stack([ls.detach(), lr_.detach(), gn.detach()]); na += 1
        if (s + 1) % LOG_N == 0:
            a = (acc / na).tolist(); acc.zero_(); na = 0
            el = time.time() - t0; ips = (s + 1 - start) / el; eta = (STEPS - s - 1) / ips
            sp = "  ⚠ gradient spike" if gem is not None and a[2] > 3 * gem else ""
            gem = a[2] if gem is None else 0.8 * gem + 0.2 * a[2]
            print(f"[{(s + 1) / STEPS * 100:5.1f}%] step {s + 1:>5}/{STEPS} | {ftime(el)} elapsed, ETA {ftime(eta)} | "
                  f"filter loss {a[0]:.3f}{arr(a[0], prev and prev[0])} | codec loss {a[1]:.3f}{arr(a[1], prev and prev[1])} | "
                  f"gnorm {a[2]:.2f} | lr {sch.get_last_lr()[0]:.1e}{sp}")
            prev = a
        if (s + 1) % EV_EVERY == 0:
            r = evl(model); model.train()
            print(f"   ── validation at step {s + 1} ({EV_N} scenes) ──"); say(r, ind="      ")
        if (s + 1) % SV_EVERY == 0:
            save_all(s + 1); print(f"   · saved ckpt, weights, embeddings (step {s + 1}) -> {OUT}/")
    save_all(STEPS, final=True)
    log(f"training done in {ftime(time.time() - t0)}")
    ls_out()

train()

### Saved files check
`checkpoint.pt` resume state, `weights_*.pt` state_dicts, `onboard_weights.pt` + `ground_decoder.pt` split, `embeddings/emb_step*.pt` probe-scene snapshots.

In [ ]:
def chk():
    from glob import glob
    if not os.path.exists(f"{OUT}/weights_latest.pt"): print("nothing saved yet"); return
    w = torch.load(f"{OUT}/weights_latest.pt", map_location="cpu")
    m2 = Vantage(CFG).to(dev); m2.load_state_dict(w["state_dict"]); m2.eval(); model.eval()
    with torch.no_grad():
        cur, ref, _ = make_batch(2, CFG, new_gen(99))
        diff = (model.score(cur, ref)[0] - m2.score(cur, ref)[0]).abs().max().item()  # should be 0
    log(f"weights_latest.pt (step {w['step']}) reload: max score diff {diff:.1e}")
    on, gr = (torch.load(f"{OUT}/{f}.pt", map_location="cpu") for f in ("onboard_weights", "ground_decoder"))
    m3 = Vantage(CFG).to(dev); m3.load_state_dict({**on["state_dict"], **gr["state_dict"]})
    n = lambda d: sum(v.numel() for v in d["state_dict"].values() if v.dtype.is_floating_point and v.dim() > 0)
    log(f"onboard + ground reassemble ok (elems: onboard {n(on):,}, ground {n(gr):,})")
    fs = sorted(glob(f"{OUT}/embeddings/emb_step*.pt"))
    e = torch.load(fs[-1], map_location="cpu")
    log(f"{len(fs)} emb snapshot(s); latest step {e['step']}: " + ", ".join(f"{k} {tuple(v.shape)}" for k, v in e.items() if torch.is_tensor(v)))
    print("   |e_cur - e_ref|: event tiles vs others, per snapshot")
    for f in fs:
        e = torch.load(f, map_location="cpu"); ev = e["labels"].bool(); dl = (e["e_cur"] - e["e_ref"]).norm(dim=-1)  # dl: [S,T]
        if ev.any(): print(f"      step {e['step']:>6}: events {dl[ev].mean():8.2f} | others {dl[~ev].mean():8.2f} | ratio {dl[ev].mean() / dl[~ev].mean():6.1f}x")

chk()

## 5. Evaluate

In [ ]:
res = evl(model, n=EV_N * 2, seed=777); show(res)

In [ ]:
model.eval()
with torch.no_grad():
    cur, ref, y = make_batch(4, CFG, new_gen(7))
    lg, mask, codes = model(cur, ref, K)  # lg, mask: [4,T]
i = int(y.sum(1).argmax())  # scene with most events
g, t = CFG.g, CFG.t
p = lg.sigmoid()[i].view(g, g).cpu()  # [g,g] per-tile prob
rgb = lambda x: ((x[:3].permute(1, 2, 0) - x[:3].min()) / (x[:3].max() - x[:3].min())).cpu().numpy()  # first 3 bands
fig, ax = plt.subplots(1, 3, figsize=(15, 5))
ax[0].imshow(rgb(ref[i])); ax[0].set_title("ref")
ax[1].imshow(rgb(cur[i])); ax[1].set_title("cur (yellow=sent, red=true)")
ax[2].imshow(p, vmin=0, vmax=1); ax[2].set_title("tile p")
for j in range(CFG.T):
    gy, gx = divmod(j, g)
    for on_, c, ls_ in [(mask[i, j].item(), "yellow", "-"), (y[i, j].item(), "red", "--")]:
        if on_: ax[1].add_patch(plt.Rectangle((gx * t - .5, gy * t - .5), t, t, fill=False, ec=c, lw=2, ls=ls_))
for a in ax: a.axis("off")
plt.tight_layout(); plt.show()
del cur, ref

## 6. Walkthrough

In [ ]:
@torch.no_grad()
def walk(model, seed=7):
    model.eval(); cfg = model.cfg; T, d, g = cfg.T, cfg.d, cfg.g
    cur, ref, y = make_batch(4, cfg, new_gen(seed))
    i = int(y.sum(1).argmax()); cur, ref, y = cur[i:i + 1], ref[i:i + 1], y[i]  # one scene: [1,C,H,H], y: [T]
    ev = y.bool(); nev = int(ev.sum())
    raw = cfg.C * cfg.H ** 2 * RAW_B / 8  # raw bytes
    print(f"SCENE      {nev} event(s) in {T} tiles; raw {raw / 1e6:.1f} MB\n")

    tc, tr = to_tiles(cur, cfg), to_tiles(ref, cfg)  # [T,C,t,t]
    print(f"1 CUT      {T} tiles {cfg.t}x{cfg.t}x{cfg.C}   {tuple(tc.shape)}")

    ec, er = model.enc(tc).view(1, T, d), model.enc(tr).view(1, T, d)  # [1,T,d]
    print(f"2 DESCRIBE {d}-d embed per tile   {tuple(ec.shape)}")

    dl = (ec - er).norm(dim=-1)[0]  # [T]
    print(f"3 COMPARE  |e_cur - e_ref|: events {dl[ev].mean():.2f} vs others {dl[~ev].mean():.2f}  ({dl[ev].mean() / dl[~ev].mean():.1f}x)")

    h0 = model.pos(model.fuse(torch.cat([ec, er, ec - er], -1))); h = model.ctx(h0)  # [1,T,d]
    cos = F.cosine_similarity(h, h0, dim=-1).mean()
    print(f"4 CONTEXT  cos(h, h0) = {cos:.2f}  (1 = attention changed nothing)")

    lg = model.head(h).squeeze(-1); p = lg.sigmoid()[0]  # lg: [1,T]; p: [T]
    top = p.topk(5)
    print("5 SCORE    top tiles (row,col): " + ", ".join(
        f"({j // g},{j % g}) p={v:.2f}{'*' if ev[j] else ''}" for v, j in zip(top.values.tolist(), top.indices.tolist())) + "   (* = event)")

    m = Vantage.select(lg, K)[0]  # [T] bool
    hit, nthr = int((m & ev).sum()), int((m & (p > 0.5)).sum())
    print(f"6 SELECT   top-{K} caught {hit}/{nev}; {nthr} tile(s) pass p>0.5")

    used = (m & (p > 0.5)).nonzero().flatten()
    note = ""
    if len(used) == 0: used, note = p.topk(1).indices, " (none passed cutoff, showing top tile)"
    x = tc[used]; q = model.codec.q(model.codec.enc(x)); L = model.codec.q.L  # x: [n,C,t,t]; q: [n,lat]
    ints = torch.round((q + 1) / 2 * L).long(); rec = model.codec.dec(q)  # ints: [n,lat] in 0..L
    r2 = 1 - ((rec - x) ** 2).mean((1, 2, 3)) / x.var(dim=(1, 2, 3))  # [n]
    print(f"7 COMPRESS {len(used)} tile(s){note}: {x[0].numel():,} -> {cfg.lat} ints in 0..{L} -> {cfg.lat * cfg.bits} bits ({cfg.lat * cfg.bits // 8} B) each")
    print(f"           first code: {ints[0, :8].tolist()} ...")
    print(f"           decoded R2: " + ", ".join(f"{v:.0%}" for v in r2.tolist())
          + ("  (negative = undertrained)" if (r2 < 0).any() else ""))

    ib = math.ceil(math.log2(T)); sb = len(used) * (cfg.lat * cfg.bits + ib) / 8  # idx bits; sent bytes
    print(f"\nSENT       {sb:,.0f} B vs {raw:,.0f} B raw -> {raw / sb:,.0f}x less ({len(used)} tile(s) x {cfg.lat * cfg.bits + ib} bits incl. {ib}-bit position)")

walk(model)

## 7. Compression check

In [ ]:
def wparams(m): return [(n, p) for n, p in m.named_parameters() if isw(n, p)]

def qw(model, bits):  # simulated symmetric per-out-channel weight quant
    m, qm = copy.deepcopy(model), 2 ** (bits - 1) - 1
    ct = {f"{n}.weight" for n, mod in m.named_modules() if isinstance(mod, nn.ConvTranspose2d)}
    for n, p in wparams(m):
        ax = 1 if n in ct else 0           # out-channel axis (convT: dim 1)
        w = p.data.movedim(ax, 0)          # [out,...]
        st = (w.flatten(1).abs().amax(1).clamp(min=1e-8) / qm).view(-1, *[1] * (w.dim() - 1))  # per-channel step
        p.data = ((w / st).round().clamp(-qm, qm) * st).movedim(0, ax).contiguous()
    return m

def pw(model, sp):  # global magnitude pruning, sp = fraction zeroed
    m = copy.deepcopy(model)
    aw = torch.cat([p.data.abs().flatten().cpu() for _, p in wparams(m)])  # all |w|
    thr = aw.kthvalue(int(sp * aw.numel())).values.item()
    for _, p in wparams(m): p.data *= (p.data.abs() > thr)
    return m

vs = [("fp32", lambda: model)] + [(f"int{b}", lambda b=b: qw(model, b)) for b in (8, 6, 4)] \
   + [(f"prune {int(s * 100)}%", lambda s=s: pw(model, s)) for s in (0.3, 0.5)]
log(f"{len(vs)} variants x {EV_N} scenes")
print(f"{'variant':<12}{'recall@k':>10}{'+thr':>8}{'recon R2':>10}")
r0 = None
for nm, build in vs:
    r = evl(build())
    r0 = r0 or r
    v = "" if nm == "fp32" else ("  OK" if r["rk"] >= r0["rk"] - 0.02 and r["r2"] >= r0["r2"] - 0.05 else "  degraded")
    print(f"{nm:<12}{r['rk']:>10.3f}{r['rkt']:>8.3f}{r['r2']:>10.3f}{v}")
print(f"\nonboard MB   " + "  ".join(f"{nm} {nob * b / 8 / 1e6:.2f}" for nm, b in [("fp32", 32), ("fp16", 16), ("int8", 8), ("int4", 4)]))
print("OK = recall within 0.02 and R2 within 0.05 of fp32")
del vs